In [0]:
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]
df=spark.createDataFrame(data,columns)
df.display()

In [0]:
df.write.saveAsTable("flipkart_catalog.bronze.orders")

In [0]:
from pyspark.sql.functions import col

df = df.filter(col("amount").isNotNull() & col("city").isNotNull())

In [0]:
df = df.withColumn('amount', col('amount').cast('int')) \
       .withColumn('date', col('date').cast('date'))

In [0]:
df=df.filter(col('amount')>0)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window = Window.partitionBy("order_id").orderBy(col("date").desc())

df = df.withColumn("rn", row_number().over(window)) \
       .filter(col("rn") == 1) \
       .drop("rn")

In [0]:
df.display()

In [0]:
df.write.mode("overwrite").saveAsTable("flipkart_catalog.silver.orders")

### GOLD

In [0]:
df = spark.read.table("flipkart_catalog.silver.orders")

In [0]:
from pyspark.sql.functions import sum

df_product = df.groupBy("product") \
               .agg(sum("amount").alias("total_sales"))
df_product.display()

In [0]:
df_category = df.groupBy("category") \
                .agg(sum("amount").alias("total_sales"))
df_category.display()

In [0]:
df_city = df.groupBy("city") \
            .agg(sum("amount").alias("total_sales"))
df_city.display()

In [0]:
df_orders=df.groupBy('customer_id').count()
df_orders.display()

In [0]:
from pyspark.sql.functions import sum
df_total_spend_customer=df.groupby('customer_id').agg(sum('amount').alias('total_spend'))
df_total_spend_customer.display()


In [0]:
df_total_spend_customer.orderBy('total_spend',ascending=False).display()

In [0]:
df_product.write \
  .mode("overwrite") \
  .saveAsTable("flipkart_catalog.gold.sales_by_product")

In [0]:
df_category.write \
  .mode("overwrite") \
  .saveAsTable("flipkart_catalog.gold.sales_by_category")

In [0]:
df_city.write \
  .mode("overwrite") \
  .saveAsTable("flipkart_catalog.gold.sales_by_city")